<a href="https://colab.research.google.com/github/tuankhoin/AI-Project-B/blob/master/Week_7_Graphic_Representations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ho Chi Minh University of Technology (HCMUT)

CO3057 - Digital Image Processing and Computer Vision

# Week 7 - Texture, Shapes, Neural Radiance Fields and Gaussian Splatting


<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/w7tattoo.jpg" height=300>

![](https://raw.githubusercontent.com/travisddavies/computer_vision_notes/refs/heads/main/Images/shape_and_texture.png?raw=true)

## Learning Outcomes
- Explain parametric and non-parametric methods for representing and synthesising texture
- Explain common applications of texture synthesis
- Explain an algorithm for texture transfer (Neural Style Transfer)
- Explain what a topological skeleton is and how it is computed
- Explain a method for detecting shape contour (active contours/snakes)
- Explain mechanisms of NeRF and Gaussian Splatting, as well as defining its pros and cons.

---
# 1. Texture
---

- A definition from image processing:
	Texture is a region with **spatial stationary** (**same statistical properties** everywhere in the region)
- A definition from computer graphics:
	Texture is 2D surface applied to a 3D model

## Types of Texture
- Periodic texture - has a subregion that repeats in a regular pattern

![](https://raw.githubusercontent.com/travisddavies/computer_vision_notes/refs/heads/main/Images/periodic_texture.png?raw=true)

- Stochastic (aperiodic) texture - generated by a random process

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/aperiodic_texture.png?raw=true)

## Texture Models (Representations)
- **Parametric** models: represent texture with a set of adjustable parameters

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/parametric_texture.png?raw=true)

- **Non-parametric** (stitching) models: represent texture as image patches

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/nonparametric_texture.png?raw=true)

## Why Model Texture?
- Texture synthesis - create more of a texure
	- Textures for computer graphics, video games, etc.
	- Image inpainting
- Texture transfer
	- Artistic effects
	- Online shopping

---
# 2. Texture Synthesis Methods
---
## 2.1 Image Stitching
Look at a neighbourhood around this patch and find similar neighbourhoods in other parts of the texture

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/similar_neighbourhood_pattersn.png?raw=true) (Original patch: ![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/original_patch.png?raw=true))

Select the most similar patches and combine:

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/neighbourhood_of_Cracks.png?raw=true) ⟹ ![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/most_probable_patch.png?raw=true)

## 2.2 Non-Parametric Texture Synthesis
1. Randomly sample a small (e.g., 3x3 pixel) patch from the original image
2. Spiral outward, filling in missing pixels by finding similar neighbourhoods in the original texture
(Neighbourhood size is a free parameter that specifies how stochastic the texture is)

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/non_parametric_texture_synthesis.png?raw=true)

Picking window size is important.

<img src="https://github.com/travisddavies/computer_vision_notes/blob/main/Images/neighbourhood_size.png?raw=true" height=220>



## 2.3 Image Quilting
"Corrupt Professor's Algorithm"
- Plagiarise as much of the source image as you can
- Then try to cover up the evidence

Algorithm:
- Choose patch and overlap size
- Initialise with a random patch
- For each subsequent patch $x$:
	- In original texture: find **most similar** patch to $x$, considering only the pixels in the **overlap** region
	- **Paste** patch, and handle overlaps

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/overlapping.png?raw=true) ![](https://upload.wikimedia.org/wikipedia/commons/b/bc/Imagequilting.gif)

How to handle overlaps: use **Graph Cuts**
- Represent neighbouring pixels as a graph
- Edge weight = overlap error
- Problem: Find path through graph with minimum total overlap error

<img src="https://github.com/travisddavies/computer_vision_notes/blob/main/Images/image_quilting_process.png?raw=true" height=220> ![](https://prateekvjoshi.com/wp-content/uploads/2013/06/cut.jpg)

## Fail cases
<img src="https://github.com/travisddavies/computer_vision_notes/blob/main/Images/image_quilting_results.png?raw=true" height=220>

<img src="https://github.com/travisddavies/computer_vision_notes/blob/main/Images/quilting_results3.png?raw=true" height=220>

<img src="https://github.com/travisddavies/computer_vision_notes/blob/main/Images/seagulls_patched.png?raw=true" height=220>


## 2.4 Image Inpainting
- Similar idea to fill in missing regions of an image:
	- Find a similar patch in _another_ image
	- Paste in patch with an error-minimising cut
- Also notice the images on the side, we will be taking cuts from these images for the one that is the most similar and blend it in this patch.

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/image_inpainting.png?raw=true)

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/w7inpaint.jpg" height=300>

## 2.5 Parametric Texture Synthesis
- Alternative to stitching approaches: represent texture with a number of parameters
- To synthesise texture, coerce a noise image to match the required parameters (usually through gradient descent)
- What parameters are needed to define a texture?

### Fourier Magnitude
We can use the Fourier transform to get parameters for texture

Reminder:
- Magnitude represents the statistical representation
- Phase represents the structure

Mechanism:
- Synthesise texture by matching Fourier magnitude
- Okay results for some simple textures, but doesn't work well in general

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/fourier_texture_synthesis.png?raw=true)

## Distribution
- Textures could be defined as a distribution over simple features, like colour and edge orientation at various scales
- Synthesise texture by matching the distribution

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/colours_and_edges.png?raw=true)

<img src="https://github.com/travisddavies/computer_vision_notes/blob/main/Images/distribution_matching_results.png?raw=true" height=220>

<img src="https://github.com/travisddavies/computer_vision_notes/blob/main/Images/more_complex_solutions.png?raw=true" height=220>

- Simple distributions of features are not sufficient
- Also need to represent feature co-occurrence
- For structured images it doesn't perform very well

## Neural Networks Response
- The set of statistics needed to represent real images may be very complex
- Instead of modelling statistics by hand, represent texture as the feature response in the layers of a trained neural network.
- Texture is represented as the correlations between feature maps at a layer of the neural network.

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/even_more_complex_statistics.png?raw=true)

## Summary
- Non-parametric texture synthesis is based on copying texture patches
	- Works very well on periodic textures
	- Disadvantage: No model of texture parameters
- Parametric textures work better on stochastic textures
	- Most methods work better on stochastic textures
	- Disadvantage: Even very complex models (e.g., based on neural networks) may be incomplete

---
# 3. Shape
---

![](https://images7.memedroid.com/images/UPLOADED368/63f5edee4b987.jpeg)

- Models of 2D shape are usually based on either:
	- The bounding contour of the shape (segments, angles)
	- The internal structure of the shape (branches)

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/models_of_shapes.png?raw=true)

---
# 4. Shape Skeletons
---

- Topological skeleton = thinnest possible version of a shape
- Formed of lines that are equidistant from the boundaries of the shape

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/shape_skeletons.png?raw=true)

- The skeleton points are the centrepoints of the largest discs that can be fit inside the shape
- If the shape was painted with a circular brush (of variable radius), the skeleton would be the path of the brush

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/rename_geometrical_description.png?raw=true)

## Skeletonisation Algorithm
- Grassfire transform - algorithm for shrinking or thinning a shape
- For each pixel within the shape, compute distance to closest boundary; **peaks** in the distance map are the skeleton

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/skeletonisation_algorithm.png?raw=true)


## Summary
- Shape skeletons represent the internal structure of shapes
- Skeleton representations work well to model shapes that have skeleton-like structure
	- Human/animal figure
	- Written characters
	- Paths/networks (e.g., city roads, blood vessels, also used in path planning in robotics)

---
# 5. Contour Representations
---

## Active Contours (Snakes)
- Parametric model that fits itself to object boundary
- "Shrink wraps" around object to capture shape

![](https://www.inf.u-szeged.hu/ssip/2007/teams/team6/gvfsnake1.gif)

## Active Contour Algorithm
1. Initialise contour outside object boundary
2. On each step, allow each point on the contour to shift 1 pixel in any direction:
	- Shift to minimise this loss function:
$$
E_{total} = \alpha E_{elasticity} + \beta E_{stiffness} + E_{edge}
$$
3. Repeat until loss does not change

Notations:
- $E_{elasticity}$ is based on contour length, penalises longer contours
- $E_{stiffness}$ is based on contour curvature, lowest for straight contour segments
- $E_{edge}$ is based on image gradient at contour locations, lowest where image gradient is highest
- $\alpha, \beta$ are free parameters

## Application: Segmentation
- Active contours are used for segmentation and tracking, particularly in medical image analysis

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/application_segmentation.png?raw=true)

## Drawbacks to Active Contours
- Requires initialisation (often from a human annotator)
- May not fit shape correctly
	- Trade-off between elasticity/smoothness and edge-matching - may fail to fit concavities in complex shapes
	- Difficult to detect shapes in clutter

![](https://github.com/travisddavies/computer_vision_notes/blob/main/Images/drawbacks_to_active_contours.png?raw=true)

## Summary
- Active contours fit a shape boundary
- Tries to find an optimal shape which is both well-fit to the edges and fairly simple (smooth, compact)
- Works well to segment objects with uniform appearance, moving objects


---
# 6. Neural Scene Representations (NeRF & Gaussian Splatting)
---

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/w7nerf.png" height=300>

## Motivation

- What we have learned so far: Traditional approaches model these properties using explicit representations such as contours, skeletons, meshes, or texture patches.
- Recent advances in computer vision and graphics attempt to **learn a full 3D representation of a scene directly from images**.

Goal:
> Given a set of images of a scene from different viewpoints, reconstruct a representation that allows us to **render the scene from any new viewpoint**.

This problem is known as **Novel View Synthesis**.

Applications include:
* Virtual reality and augmented reality
* Film and visual effects
* Robotics and autonomous navigation
* 3D scanning and digital twins

Two important modern approaches are:
* **Neural Radiance Fields (NeRF)**
* **3D Gaussian Splatting**

Both represent scenes as **continuous volumetric models** instead of meshes or point clouds.


---
# 7. Neural Radiance Fields (NeRF)
---

![](https://visionbook.mit.edu/figures/nerfs/flatland_cameras_and_images.png)

NeRF represents a 3D scene as a **continuous function** implemented by a neural network.

Instead of explicitly storing geometry, the scene is represented as

$$
F_\Omega(p,d) = (c,\sigma)
$$

where

* $p=(x,y,z)$ : spatial location in 3D
* $d=(θ,𝛷)$ : viewing direction
* $c=(r,g,b)$ : emitted colour
* $\sigma$ : volume density (opacity/transparency)
* $F_\Omega$ (scene representation) is a mapping function, usually a neural network or Gaussian mixture (differentiable for optimization). $\Omega$ denotes the parameters.

Thus the network models how **light is emitted and absorbed in the scene**.

---
# 8. Rendering from NeRF
---

## Pixel ray
![](https://visionbook.mit.edu/figures/nerfs/flatland_volume_rendering.png)

Each pixel corresponds to a **camera ray**
$
r(t) = o + td
$

where

* $o$ = camera origin
* $d$ = ray direction
* $t$ = distance along the ray

A ray returns the first color that it reflects (ray casting) ⇒ We try to collect:
- Points are sampled along this ray $x_i = r(t_i)$
- For each sampled point the network predicts $(c_i, \sigma_i)$

## Volume Rendering Algorithm

<img src="https://visionbook.mit.edu/figures/nerfs/flatland_training.jpg" height=500>

Each sampled point contributes colour depending on its density.

- Opacity of each sample: $\alpha_i = 1 - e^{-\sigma_i \delta_i}$
  - $\delta_i$ is the spacing between samples.
- Transmittance/Transparency: $T_i = \prod_{j<i} (1 - \alpha_j)$

The final rendered pixel colour $C$ given point colors $c_i$ is

$$
C(r) = \sum_i T_i \alpha_i c_i
$$

- $ T_i \alpha_i$ can be treated as weights ⟶ Probability distribution

## Loss function for model training

Predicted colours are compared to the true image colours.

Loss function:

$$
L = \sum ||C(r) - C_{true}||
$$

The neural network parameters $\theta$ are optimized using gradient descent.

After training, the model can generate **novel views of the scene**.

## Characteristics of NeRF

Advantages

* extremely realistic rendering
* continuous scene representation
* captures view-dependent effects (reflections, lighting)

Limitations

* expensive training
* slow rendering
* requires many input images



---
# 9. Gaussian Splatting
---

![](https://preview.redd.it/i-just-tried-this-and-its-true-v0-5oqqn2l5vgfg1.jpeg?width=640&crop=smart&auto=webp&s=cb339b3464491e0240232c39eb856034b69f7d5a)

## Motivation

NeRF achieves high quality results but is **computationally expensive**.

Gaussian Splatting replaces the neural field with an **explicit volumetric representation** (ellipsoid) consisting of many 3D Gaussian primitives.


## Gaussian Representation

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/w7gaussianblob.png" height=300>

Each Gaussian is defined by

* mean position $\mu$
* covariance matrix $\Sigma$
* colour $c$
* opacity $\alpha$

The Gaussian density function is

$$
G(x) = e^{-(x-\mu)^T \Sigma^{-1} (x-\mu)}
$$

A scene is therefore represented as

$$
S = {G_1, G_2, ..., G_N}
$$

where each $G_i$ is a Gaussian primitive.


## Rendering

![](https://blog.42yeah.is/assets/post_assets/gsr/cleanup_2.jpg)

Rendering involves projecting the Gaussians into the camera view.

Steps:

1. Transform Gaussians to camera coordinates
2. Project them onto the image plane
3. Render each Gaussian as a **2D ellipse** (see picture above)
4. Blend colours using alpha compositing (sorting colors by depth)

Pixel colour is computed as $C = \sum_i w_i c_i$, similar to volumetric rendering.

Because Gaussians are explicit primitives, this process can be implemented efficiently on GPUs.

Further reading: [this blog](https://blog.42yeah.is/rendering/opengl/2023/12/20/rasterizing-splats.html)

## Characteristics of Gaussian Splatting

Advantages

* real-time rendering (often $60+$ FPS)
* high reconstruction quality
* significantly faster than NeRF
* No neural networks

Limitations

* large memory usage (many Gaussians), i.e. still needs GPU!
* training still requires optimization
* less flexible than neural representations for some effects

"No deep learning. Why GPU?"
- Tiling: Split the image in 16x16 Tiles – helps threads to work collaboratively
- Global sort: sort the primitives when doing color blending



---
# 10. More details
---

## Comparison

| Method             | Representation             | Rendering Speed | Quality   |
| ------------------ | -------------------------- | --------------- | --------- |
| Mesh models        | polygons + textures        | very fast       | moderate  |
| NeRF               | neural volumetric function | slow            | very high |
| Gaussian splatting | explicit Gaussians         | real-time       | high      |

These approaches can be seen as **parametric scene models**, similar in spirit to the parametric texture models discussed earlier.

## Current Research Directions

### Faster Training for NeRF

Methods such as **Instant-NGP** use multi-resolution hash encoding to accelerate NeRF training ⟶ Training time reduced from hours to minutes.

### Sparse-View Reconstruction

Standard NeRF requires many images ⟶ New approaches reconstruct scenes from only a few images.

Examples
* PixelNeRF
* RegNeRF

### Dynamic Scenes

Original NeRF assumes static scenes ⟶ Extensions model time to $F(x,d,t)$

Examples
* D-NeRF
* HyperNeRF

### Vaired condition samples (internet images)
Current NeRF assumes similar conditions like lighting ⟶ Introduced per-image appearance embeddings and transient uncertainty fields, handling lighting changes and occlusions

Paper: NeRF-W

### Improved Geometry

Some methods incorporate **signed distance functions (SDF)** to better model surfaces. Example: **NeuS**

### Integration with other data types
- Human motion: HumanNeRF
- Camera calibration: SC-NeRF


# Summary

Neural scene representations extend traditional shape and texture modelling by learning **volumetric 3D representations from images**.

NeRF

* represents scenes as neural radiance functions
* produces very high quality renderings
* computationally expensive

Gaussian Splatting

* represents scenes using explicit 3D Gaussian primitives
* enables real-time rendering
* maintains high reconstruction quality

These techniques represent a major step toward **learning full 3D models of the visual world from images**.
